# Single-Asset Timing with Trend Pullbacks and vectorbt

This notebook builds a single-asset timing signal: trade long when the trend is positive and the residual is temporarily cheap. It runs through a transparent pandas backtest first and then shows how to route the same signal to vectorbt.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in [ROOT / "src", ROOT / "examples"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant_trading.data import fetch_yahoo_prices, fetch_yahoo_ohlcv, data_audit_report, DEFAULT_UNIVERSES
from quant_trading.features import decompose_one_series, walkforward_decompose, build_feature_table
from quant_trading.signals import (
    trend_pullback_signals,
    residual_mean_reversion_signals,
    turtle_donchian_signals,
    pair_trading_weights,
    cross_sectional_rotation_weights,
    residual_stress_filter,
)
from quant_trading.backtest import backtest_weights, backtest_long_short_signals, summarize_returns

In [ ]:
prices = fetch_yahoo_prices(["SPY", "QQQ"], start="2016-01-01", cache_dir=ROOT / "examples" / "quant_trading" / "data" / "cache")
features = walkforward_decompose(prices, method="STL", period=63, train_window=252, step=21)
entries, exits = trend_pullback_signals(prices, features, residual_entry_z=-1.0, residual_exit_z=0.25)
result = backtest_long_short_signals(prices, entries, exits, fee_bps=1.0, slippage_bps=2.0)
result.stats_frame()

In [ ]:
ax = result.equity.plot(figsize=(10, 4), title="Trend-pullback timing equity curve")
ax.set_xlabel("date")
plt.show()

## Optional: vectorbt adapter

Install vectorbt first if needed. The same entry/exit matrices can be passed to `Portfolio.from_signals` through the adapter.

In [ ]:
from quant_trading.frameworks import run_vectorbt_from_signals

# portfolio = run_vectorbt_from_signals(prices, entries, exits, fees=0.0001, slippage=0.0002)
# portfolio.stats()